In [36]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [37]:
#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

import os

current_directory = os.getcwd()
print(f"Current directory: {current_directory}")
if current_directory.endswith("RecSys_Course_AT_PoliMi"):
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/RecSys_Course_AT_PoliMi")

#!pwd
!python run_compile_all_cython.py

Current directory: /Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/Results
zsh:1: command not found: python


In [38]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt
from Recommenders.MatrixFactorization.PureSVDRecommender import ScaledPureSVDRecommender

In [39]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

In [40]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [41]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))

In [42]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

[<Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>,
 <Compressed Sparse Row sparse matrix of dtype 'float64'
 	with 608611 stored elements and shape (27095, 6969)>]

In [ ]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

                          
    start_time = time.time()
    scores = []
    for i in range(5):
        URM_combined = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        
        
        #Cambiare il modello qui sotto, insieme al range e ai parametri 
        recommender_instance = ScaledPureSVDRecommender(URM_combined)

        recommender_instance.fit(
        # Integer parameter: Number of latent components
        # Range: Typically between 10 and 500 depending on dataset size
        num_factors = optuna_trial.suggest_int("num_factors", 10, 200),

        # Float parameter: Item scaling
        # Range: 0.0 (binary) to 1.0 (linear count). 
        # Sometimes going slightly above 1.0 helps, but 0-1 is standard.
        scaling_items = optuna_trial.suggest_float("scaling_items", 0.0, 0.04),

        # Float parameter: User scaling
        # Mitigates the impact of very active users
        # users with thousands of interactions and users with only a few interactions are now treated more equally
        scaling_users = optuna_trial.suggest_float("scaling_users", 0.4, 0.7),
        
        # FIXED PARAMETER: Do not tune this. Keep it constant for reproducibility.
        random_seed = 41
    )
        
        evaluator_test = EvaluatorHoldout(URM_parts[i], cutoff_list=[20])
        result, _ = evaluator_test.evaluateRecommender(recommender_instance)
        #print("prova = ", result["MAP"].values[0])
        #print(result)
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [44]:
import optuna
optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 100)

[I 2025-11-25 22:55:15,692] A new study created in memory with name: no-name-0eef9cca-8895-44d3-aaee-c669ccc14a67


ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 3.59 sec
EvaluatorHoldout: Ignoring 34 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27061 (100.0%) in 8.18 sec. Users per second: 3306
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 3.33 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 8.14 sec. Users per second: 3323
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 3.33 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 8.24 sec. Users per second: 3282
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done

[I 2025-11-25 22:56:14,104] Trial 0 finished with value: 0.20411257825781118 and parameters: {'num_factors': 168, 'scaling_items': 0.04428786444137992, 'scaling_users': 0.7960241943878996}. Best is trial 0 with value: 0.20411257825781118.


[0.20422979521090986, 0.2040054333491894, 0.20360551838583324, 0.20462530430521336, 0.20409684003791007]
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 1.25 sec
EvaluatorHoldout: Ignoring 34 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27061 (100.0%) in 7.85 sec. Users per second: 3446
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 1.24 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 7.72 sec. Users per second: 3505
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 1.75 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 7.86 sec. Users per second: 3441
ScaledPureSVD

[I 2025-11-25 22:57:00,380] Trial 1 finished with value: 0.1983231819639999 and parameters: {'num_factors': 57, 'scaling_items': 0.07081844437451394, 'scaling_users': 0.7485026331886775}. Best is trial 0 with value: 0.20411257825781118.


...

[0.21151262627973272, 0.2128530332845266, 0.21194535871601575, 0.2121348882302302, 0.21175286246829136]
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 2.29 sec
EvaluatorHoldout: Ignoring 34 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27061 (100.0%) in 7.74 sec. Users per second: 3497
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 2.25 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 7.85 sec. Users per second: 3447
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 2.28 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 7.80 sec. Users per second: 3469
ScaledPureSVDR

[I 2025-11-26 01:08:59,787] Trial 98 finished with value: 0.2163083508944812 and parameters: {'num_factors': 107, 'scaling_items': 0.0051694696632211354, 'scaling_users': 0.5400090207020097}. Best is trial 57 with value: 0.21725345726423523.


[0.21639314815036192, 0.2168939795227021, 0.21626221252992284, 0.21669384415365026, 0.2152985701157689]
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 1.99 sec
EvaluatorHoldout: Ignoring 34 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27061 (100.0%) in 7.79 sec. Users per second: 3476
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 1.91 sec
EvaluatorHoldout: Ignoring 33 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27062 (100.0%) in 7.80 sec. Users per second: 3471
ScaledPureSVDRecommender: Computing SVD decomposition...
ScaledPureSVDRecommender: Computing SVD decomposition... done in 1.95 sec
EvaluatorHoldout: Ignoring 35 ( 0.1%) Users that have less than 1 test interactions
EvaluatorHoldout: Processed 27060 (100.0%) in 7.91 sec. Users per second: 3421
ScaledPureSVDR

[I 2025-11-26 01:09:49,239] Trial 99 finished with value: 0.21684349672559905 and parameters: {'num_factors': 96, 'scaling_items': 0.002145346614556369, 'scaling_users': 0.555893691550176}. Best is trial 57 with value: 0.21725345726423523.


[0.2163571545329787, 0.21871503188696925, 0.21602723487341632, 0.21707583825345686, 0.21604222408117413]


In [45]:
optuna_study.best_trial.params

{'num_factors': 101,
 'scaling_items': 0.004145476452561414,
 'scaling_users': 0.5265438755286422}

In [ ]:
import os

current_directory = os.getcwd()
if current_directory.endswith("Results"):
    print("In the correct directory.")
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/Results")

# 1. Get the dataframe from the study
df_trials = optuna_study.trials_dataframe()

# 2. Filter for the columns specific to ScaledPureSVDRecommender
try:
    df_filtered = df_trials[[
        'number', 
        'params_num_factors',
        'params_scaling_items', 
        'params_scaling_users',
        'value'
    ]].copy()
except KeyError as e:
    print(f"Error: One or more parameters not found in study. Check your objective function names. {e}")
    # Fallback if you haven't run the study yet or names differ
    df_filtered = df_trials.copy() 

# 3. Rename the columns to be CSV-friendly
df_filtered.columns = ['trial_id', 'num_factors', 'scaling_items', 'scaling_users', 'recall']

# 4. Automatic Versioning Logic
base_filename = "scaledpure_svd_results_v"
extension = ".csv"
version = 1

while True:
    csv_filename = f"{base_filename}{version}{extension}"
    if not os.path.exists(csv_filename):
        break # Trovato un nome file non ancora esistente
    version += 1

# 5. Save to CSV (Senza ordinamento)
df_filtered.to_csv(csv_filename, index=False)

print(f"✅ Full results saved to {csv_filename}")

# 6. Visualizza i Top 5 (opzionale, solo per visualizzazione rapida)
print("------------ FIRST 5 TRIALS ------")
print(df_filtered.head(5))
print("----------------------------------")

✅ Full results saved to scaledpure_svd_results_v2.csv
------------ FIRST 5 TRIALS ------
   trial_id    recall  num_factors  scaling_items  scaling_users
0         0  0.204113          168       0.044288       0.796024
1         1  0.198323           57       0.070818       0.748503
2         2  0.211216          109       0.030412       0.597086
3         3  0.191388           36       0.031967       0.754248
4         4  0.205288          103       0.059365       0.617983
----------------------------------


In [47]:
#/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

import os

current_directory = os.getcwd()
if current_directory.endswith("Results"):
    print("In the correct directory.")
    pass
else:
    os.chdir("/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/Results")

df_filtered.to_csv("scaledpureSVD_opt_results.csv", index=False)

!pwd
!python run_compile_all_cython.py

In the correct directory.
/Users/filippo/Documents/PoliMI/Recommender Systems/Challenge/Models/Results
zsh:1: command not found: python


In [48]:
optuna_study.best_value
best_index = save_results.results_df["result"].idxmax()
best_hyperparams = save_results.results_df.loc[best_index].to_dict()

del best_hyperparams["result"]
del best_hyperparams["train_time (min)"]
print("-----------best hyperparameters-----------\n")
print(best_hyperparams)

-----------best hyperparameters-----------

{}
